In [0]:
active_cmd_df = spark.sql("""
SELECT *
FROM hvacapp_dev3.serving.active_control_command
""")

display(active_cmd_df)

In [0]:
# Extract active setpoint

active_cmd = spark.sql("""
SELECT *
FROM hvacapp_dev3.serving.active_control_command
LIMIT 1
""").collect()

if len(active_cmd) > 0:
    commanded_setpoint = float(active_cmd[0]["commanded_setpoint_c"])
    hvac_system_id = active_cmd[0]["hvac_system_id"]
else:
    commanded_setpoint = 9.0
    hvac_system_id = "HVAC_SYSTEM_001"

print("Active commanded setpoint:", commanded_setpoint)
print("HVAC system:", hvac_system_id)

In [0]:
from pyspark.sql import Row
from datetime import datetime
import uuid
import random
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, DoubleType

# Read sensors
sensors = spark.table("hvacapp_dev3.master.sensor").collect()

# Read latest active command
active_cmd = spark.sql("""
SELECT *
FROM hvacapp_dev3.serving.active_control_command
LIMIT 1
""").collect()

if len(active_cmd) > 0:
    commanded_setpoint = float(active_cmd[0]["commanded_setpoint_c"])
    hvac_system_id = active_cmd[0]["hvac_system_id"]
    command_id = active_cmd[0]["command_id"]
else:
    commanded_setpoint = 9.0
    hvac_system_id = "HVAC_SYSTEM_001"
    command_id = None

current_ts = datetime.now()

# Simulate system response based on command
base_supply_temp = round(commanded_setpoint + random.uniform(-0.2, 0.2), 2)
base_return_temp = round(base_supply_temp + random.uniform(2.8, 4.2), 2)

# Flow and power respond to setpoint
if commanded_setpoint <= 8.2:
    base_flow_lpm = round(random.uniform(800.0, 900.0), 2)
    base_pump_speed = round(random.uniform(78.0, 88.0), 2)
    base_power_kw = round(random.uniform(360.0, 420.0), 2)

elif commanded_setpoint >= 9.3:
    base_flow_lpm = round(random.uniform(680.0, 760.0), 2)
    base_pump_speed = round(random.uniform(65.0, 75.0), 2)
    base_power_kw = round(random.uniform(290.0, 340.0), 2)

else:
    base_flow_lpm = round(random.uniform(740.0, 820.0), 2)
    base_pump_speed = round(random.uniform(70.0, 82.0), 2)
    base_power_kw = round(random.uniform(330.0, 370.0), 2)

rows = []

for s in sensors:
    metric_name = s["metric_name"]
    sensor_id = s["sensor_id"]
    equipment_id = s["equipment_id"]
    site_id = s["site_id"]
    building_id = s["building_id"]
    unit = s["unit"]

    if metric_name == "chw_supply_temp_c":
        value = base_supply_temp
    elif metric_name == "chw_return_temp_c":
        value = base_return_temp
    elif metric_name == "hvac_power_kw":
        value = base_power_kw
    elif metric_name == "chw_flow_lpm":
        value = base_flow_lpm
    elif metric_name == "pump_speed_pct":
        value = base_pump_speed
    else:
        value = 0.0

    rows.append(
        Row(
            event_id=str(uuid.uuid4()),
            event_ts=current_ts,
            ingest_ts=current_ts,
            site_id=site_id,
            building_id=building_id,
            equipment_id=equipment_id,
            sensor_id=sensor_id,
            metric_name=metric_name,
            metric_value=float(value),
            unit=unit,
            quality_flag="GOOD",
            source_system="SIMULATOR_WITH_FEEDBACK",
            raw_payload=None
        )
    )

schema = StructType([
    StructField('event_id', StringType(), True), 
    StructField('event_ts', TimestampType(), True), 
    StructField('ingest_ts', TimestampType(), True), 
    StructField('site_id', StringType(), True), 
    StructField('building_id', StringType(), True), 
    StructField('equipment_id', StringType(), True), 
    StructField('sensor_id', StringType(), True), 
    StructField('metric_name', StringType(), True), 
    StructField('metric_value', DoubleType(), True), 
    StructField('unit', StringType(), True), 
    StructField('quality_flag', StringType(), True), 
    StructField('source_system', StringType(), True), 
    StructField('raw_payload', StringType(), True)])
    
sim_df = spark.createDataFrame(rows, schema)

display(sim_df)

In [0]:
sim_df.write.mode("append").saveAsTable("hvacapp_dev3.raw.bronze_hvac_telemetry")

In [0]:
from pyspark.sql import Row
from datetime import datetime
import uuid

feedback_rows = [
    Row(
        feedback_id=str(uuid.uuid4()),
        feedback_ts=current_ts,
        command_id=command_id,
        site_id=site_id,
        building_id=building_id,
        hvac_system_id=hvac_system_id,
        control_point_name="chw_supply_temp_setpoint_c",
        commanded_setpoint_c=float(commanded_setpoint),
        observed_supply_temp_c=float(base_supply_temp),
        observed_return_temp_c=float(base_return_temp),
        observed_power_kw=float(base_power_kw),
        feedback_status="RESPONDED",
        notes="Simulator generated telemetry based on latest applied command",
        created_ts=current_ts
    )
]

feedback_df = spark.createDataFrame(feedback_rows)
display(feedback_df)

In [0]:
feedback_df.write.mode("append").saveAsTable("hvacapp_dev3.serving.command_feedback_log")

In [0]:
%sql
SELECT *
FROM hvacapp_dev3.serving.command_feedback_log
ORDER BY feedback_ts DESC;

In [0]:
%sql
SELECT *
FROM hvacapp_dev3.serving.active_control_command;

In [0]:
%sql
SELECT *
FROM hvacapp_dev3.serving.command_feedback_log
ORDER BY feedback_ts DESC;

In [0]:
%sql
SELECT
  feedback_ts,
  commanded_setpoint_c,
  observed_supply_temp_c,
  observed_return_temp_c,
  observed_power_kw,
  feedback_status
FROM hvacapp_dev3.serving.command_feedback_log
ORDER BY feedback_ts DESC;

In [0]:
# Success criteria

# control_commands has rows
# active_control_command returns latest applied command
# new simulator telemetry changes according to setpoint
# command_feedback_log has rows
# downstream KPI tables reflect updated behavior